In [8]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: fineGrained).
The token `idl project code llama` has been saved to /root/.cache/huggingface/stored_tokens
Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to r

In [7]:
!pip install bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 95.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [6]:
!git clone https://github.com/openai/human-eval
!pip install -e human-eval

Cloning into 'human-eval'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 34 (delta 12), reused 7 (delta 7), pack-reused 8 (from 1)
Receiving objects: 100% (34/34), 55.80 KiB | 1.74 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Obtaining file:///content/human-eval
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=5d410b870fb793c30bb0eaf59314d95ab4a34bd96e18991d32ef65c8f8e9bfa6
  Stored in directory: /root/.cache/pip/wheels/46/54/24/1624fd5b8674eb1188623f7e8e17cdf7c0f6c24b609dfb8a89
Successfully built fire
  Running setup.py develop for human-eval


In [9]:
%cd human-eval

/content/human-eval


In [ ]:
from human_eval.data import read_problems, write_jsonl
import itertools
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# Step 1: Load the HumanEval problems
problems = read_problems()

# Step 2: Get the first 10 problem IDs
first_10_keys = list(itertools.islice(problems.keys(), 5))

# Step 3: Load your model and tokenizer
model_name = "meta-llama/CodeLlama-7b-Python-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to("cuda" if torch.cuda.is_available() else "cpu")

# Step 4: Generate completions for each prompt
samples = []
for problem_id in tqdm(first_10_keys):
    prompt = problems[problem_id]["prompt"]

    print(prompt)

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate text
    outputs = model.generate(
        inputs.input_ids,
        max_length=500,
        temperature=0.2,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode the generated text
    generated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Important: The completion should NOT include the prompt itself
    # We need to extract just the completion part
    completion = generated_code[len(prompt):]

    # Save the result
    samples.append({
        "task_id": problem_id,
        "completion": completion
    })

    # Print for debugging
    print(f"Problem ID: {problem_id}")
    print("=" * 40)
    print("Prompt + Completion:")
    print(generated_code)
    print("=" * 40)
    print("Just Completion:")
    print(completion)
    print("\n")

# Step 5: Save the samples to a JSONL file
write_jsonl("humaneval_samples.jsonl", samples)

# !evaluate_functional_correctness humaneval_samples.jsonl

In [38]:
from human_eval.data import write_jsonl, read_problems
import itertools

# Load all problems
problems = read_problems()

# Get the first 10 problem IDs
first_10_keys = list(itertools.islice(problems.keys(), 5))

# Create a dictionary with just those 10 problems
selected_problems = {k: problems[k] for k in first_10_keys}

write_jsonl("selected_problems.jsonl",
            [{"task_id": k, **v} for k, v in selected_problems.items()])

In [39]:
!evaluate_functional_correctness humaneval_samples.jsonl --problem_file=selected_problems.jsonl

Reading samples...
5it [00:00, 359.08it/s]
Running test suites...
100% 5/5 [00:00<00:00, 101.72it/s]
Writing results to humaneval_samples.jsonl_results.jsonl...
100% 5/5 [00:00<00:00, 12929.42it/s]
{'pass@1': 0.8}


In [35]:
!python3 results.py

In [56]:
!python main.py

Creating subset of 164 problems...
Saved 164 problems to selected_problems.jsonl
Loading model: meta-llama/CodeLlama-7b-Python-hf
2025-03-15 22:04:22.771064: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1742076262.790474   64660 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1742076262.797067   64660 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading checkpoint shards: 100% 2/2 [00:07<00:00,  3.88s/it]
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing

In [55]:
!python3 results.py

Reading results from: humaneval_samples.jsonl_results.jsonl

===== RESULTS SUMMARY =====
Total problems: 5
Passed: 4
Failed: 1
Pass rate: 80.00%

===== DETAILED RESULTS =====
HumanEval/0: ✓ PASSED
HumanEval/1: ✗ FAILED
HumanEval/2: ✓ PASSED
HumanEval/3: ✓ PASSED
HumanEval/4: ✓ PASSED


In [45]:
import gzip
import json

# Path to the gzipped file
file_path = '/content/human-eval/data/HumanEval.jsonl.gz'

# Count problems
problem_count = 0
task_ids = []

# Open and read the gzipped file
with gzip.open(file_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        problem_count += 1
        if 'task_id' in data:
            task_ids.append(data['task_id'])

# Print results
print(f"Total problems in HumanEval.jsonl.gz: {problem_count}")

Total problems in HumanEval.jsonl.gz: 164


In [50]:
from human_eval.data import read_problems
import itertools

# Load all problems from the dataset
problems = read_problems()

# Get the first 5 problem IDs
first_5_keys = list(itertools.islice(problems.keys(), 5))

# Print the prompts for the first 5 problems
for problem_id in first_5_keys:
    print(f"Problem ID: {problem_id}")
    print("=" * 80)
    print(problems[problem_id]["prompt"])
    print("=" * 80)
    print("\n")

Problem ID: HumanEval/0
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """



Problem ID: HumanEval/1
from typing import List


def separate_paren_groups(paren_string: str) -> List[str]:
    """ Input to this function is a string containing multiple groups of nested parentheses. Your goal is to
    separate those group into separate strings and return the list of those.
    Separate groups are balanced (each open brace is properly closed) and not nested within each other
    Ignore any spaces in the input string.
    >>> separate_paren_groups('( ) (( )) (( )( ))')
    ['()', '(())', '(()())']
    """



Problem ID: HumanEval/2


def truncate_number(number: float) -> float:
    """ Given a posit

In [234]:
import pandas as pd

# Read the CSV file into a pandas DataFrame
df_original = pd.read_csv('/content/human-eval/humaneval_extracted.csv')

# Display the first few rows
df_original

,task_id,prompt,entry_point,canonical_solution,test
0,HumanEval/0,from typing import List\n\n\ndef has_close_ele...,has_close_elements,"for idx, elem in enumerate(numbers):\n ...","\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
1,HumanEval/1,from typing import List\n\n\ndef separate_pare...,separate_paren_groups,result = []\n current_string = []\n ...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
2,HumanEval/2,\n\ndef truncate_number(number: float) -> floa...,truncate_number,return number % 1.0\n,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
3,HumanEval/3,from typing import List\n\n\ndef below_zero(op...,below_zero,balance = 0\n\n for op in operations:\n...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
4,HumanEval/4,from typing import List\n\n\ndef mean_absolute...,mean_absolute_deviation,mean = sum(numbers) / len(numbers)\n re...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
...,...,...,...,...,...
159,HumanEval/159,"\ndef eat(number, need, remaining):\n """"""\n...",eat,if(need <= remaining):\n return [ n...,def check(candidate):\n\n # Check some simp...
160,HumanEval/160,"\ndef do_algebra(operator, operand):\n """"""\...",do_algebra,expression = str(operand[0])\n for oprt...,def check(candidate):\n\n # Check some simp...
161,HumanEval/161,"\ndef solve(s):\n """"""You are given a string...",solve,flg = 0\n idx = 0\n new_str = list(s...,def check(candidate):\n\n # Check some simp...
162,HumanEval/162,"\ndef string_to_md5(text):\n """"""\n Given...",string_to_md5,import hashlib\n return hashlib.md5(tex...,def check(candidate):\n\n # Check some simp...


In [250]:
import pandas as pd

# Read the CSV file into a pandas DataFrame
df_completions = pd.read_csv('/content/human-eval/update10.csv')

# Display the first few rows
df_completions

,task_id,completion,prompt
0,HumanEval/0,numbers_to_check = sorted(numbers)\n fo...,from typing import List\n\n\ndef has_close_ele...
1,HumanEval/1,\n parens = []\n # Stack for keeping tr...,from typing import List\n\n\ndef separate_pare...
2,HumanEval/2,return number - int(number)\n\n\ndef is_py...,\n\ndef truncate_number(number: float) -> floa...
3,HumanEval/3,balance = 0\n for operation in operatio...,from typing import List\n\n\ndef below_zero(op...
4,HumanEval/4,numbers_mean = sum(numbers) / len(numbers)...,from typing import List\n\n\ndef mean_absolute...
...,...,...,...
159,HumanEval/159,# TODO - Add Your Code Here - 8 Lines (~3-...,"\ndef eat(number, need, remaining):\n """"""\n..."
160,HumanEval/160,op=operator[0]; operand1=operand[0]; opera...,"\ndef do_algebra(operator, operand):\n """"""\..."
161,HumanEval/161,\n #for i in range (len(s)): ## Loop ...,"\ndef solve(s):\n """"""You are given a string..."
162,HumanEval/162,\n import hashlib\n\n if not text:\n ...,"\ndef string_to_md5(text):\n """"""\n Given..."


In [251]:
new_df = pd.merge(
    df_completions[['task_id', 'completion']],
    df_original[['task_id', 'prompt']],
    on='task_id',
    how='left'
)
new_df

,task_id,completion,prompt
0,HumanEval/0,numbers_to_check = sorted(numbers)\n fo...,from typing import List\n\n\ndef has_close_ele...
1,HumanEval/1,\n parens = []\n # Stack for keeping tr...,from typing import List\n\n\ndef separate_pare...
2,HumanEval/2,return number - int(number)\n\n\ndef is_py...,\n\ndef truncate_number(number: float) -> floa...
3,HumanEval/3,balance = 0\n for operation in operatio...,from typing import List\n\n\ndef below_zero(op...
4,HumanEval/4,numbers_mean = sum(numbers) / len(numbers)...,from typing import List\n\n\ndef mean_absolute...
...,...,...,...
159,HumanEval/159,# TODO - Add Your Code Here - 8 Lines (~3-...,"\ndef eat(number, need, remaining):\n """"""\n..."
160,HumanEval/160,op=operator[0]; operand1=operand[0]; opera...,"\ndef do_algebra(operator, operand):\n """"""\..."
161,HumanEval/161,\n #for i in range (len(s)): ## Loop ...,"\ndef solve(s):\n """"""You are given a string..."
162,HumanEval/162,\n import hashlib\n\n if not text:\n ...,"\ndef string_to_md5(text):\n """"""\n Given..."


In [252]:
failed_ids = df_completions[
    (df_completions['completion'].isna()) |
    (df_completions['completion'] == '') |
    (df_completions['completion'].astype(str).str.isspace()) |
    (df_completions['completion'].astype(str).str.strip() == '')
]['task_id'].tolist()  # Added ['task_id'] here

failed_questions_df = df_original[df_original['task_id'].isin(failed_ids)]
len(failed_questions_df)

5

In [253]:
failed_questions_df

,task_id,prompt,entry_point,canonical_solution,test
70,HumanEval/70,\ndef strange_sort_list(lst):\n '''\n Gi...,strange_sort_list,"res, switch = [], True\n while lst:\n ...",def check(candidate):\n\n # Check some simp...
82,HumanEval/82,"\ndef prime_length(string):\n """"""Write a fu...",prime_length,l = len(string)\n if l == 0 or l == 1:\...,def check(candidate):\n\n # Check some simp...
95,HumanEval/95,"\ndef check_dict_case(dict):\n """"""\n Giv...",check_dict_case,if len(dict.keys()) == 0:\n return ...,def check(candidate):\n\n # Check some simp...
127,HumanEval/127,"\ndef intersection(interval1, interval2):\n ...",intersection,def is_prime(num):\n if num == 1 or...,def check(candidate):\n\n # Check some simp...
153,HumanEval/153,"\ndef Strongest_Extension(class_name, extensio...",Strongest_Extension,strong = extensions[0]\n my_val = len([...,def check(candidate):\n\n # Check some simp...


In [254]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
import re
import pandas as pd

# Load the model
def load_model(model_name="meta-llama/CodeLlama-7b-Python-hf"):
    print(f"Loading model: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = "[PAD]"
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )

    model.resize_token_embeddings(len(tokenizer))

    return model, tokenizer

# Generate solutions directly from the failed_questions_df
def regenerate_solutions(failed_questions_df, model_name="meta-llama/CodeLlama-7b-Python-hf"):
    # Load model
    model, tokenizer = load_model(model_name)

    # Create results list
    results = []

    # Process each problem
    print(f"Regenerating solutions for {len(failed_questions_df)} problems...")

    for _, row in tqdm(failed_questions_df.iterrows(), total=len(failed_questions_df)):
        task_id = row['task_id']
        prompt = row['prompt']

        # Generate solution
        inputs = tokenizer(prompt, return_tensors="pt", padding=True, return_attention_mask=True).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=1024,
                temperature=0.7,
                top_p=0.95,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.2,
            )

        # Decode and extract completion
        generated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)
        completion = generated_code[len(prompt):]

        # Store result
        results.append({
            "task_id": task_id,
            "prompt": prompt,
            "completion": completion
        })

    # Convert to DataFrame
    results_df = pd.DataFrame(results)

    print(f"Generated {len(results_df)} new solutions")
    return results_df

# Use it directly
new_solutions_df = regenerate_solutions(failed_questions_df)

# Save results if needed
new_solutions_df.to_csv('regenerated_solutions.csv', index=False)

Loading model: meta-llama/CodeLlama-7b-Python-hf


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Regenerating solutions for 5 problems...


100%|██████████| 5/5 [00:26<00:00,  5.24s/it]

Generated 5 new solutions


In [255]:
new_solutions_df

,task_id,prompt,completion
0,HumanEval/70,\ndef strange_sort_list(lst):\n '''\n Gi...,# Your code here\n if len(lst) <= 0 :\...
1,HumanEval/82,"\ndef prime_length(string):\n """"""Write a fu...",#your code here\n
2,HumanEval/95,"\ndef check_dict_case(dict):\n """"""\n Giv...",\n # TODO - by the end of this lesson y...
3,HumanEval/127,"\ndef intersection(interval1, interval2):\n ...",
4,HumanEval/153,"\ndef Strongest_Extension(class_name, extensio...",


In [ ]:
new_solutions_df

In [62]:
import gzip
import json
import pandas as pd

# Path to the gzipped JSONL file
file_path = '/content/human-eval/data/HumanEval.jsonl.gz'

# Extract the data and convert to a DataFrame
data = []
with gzip.open(file_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

# Convert to DataFrame for easier analysis
df = pd.DataFrame(data)

# Display basic information
print(f"Total records: {len(df)}")
print("\nColumns in the dataset:")
print(df.columns.tolist())

# Display the first few records
print("\nSample data:")
print(df.head())

# Optionally save as a CSV for easier viewing
df.to_csv('humaneval_extracted.csv', index=False)
print("\nData extracted and saved to 'humaneval_extracted.csv'")

Total records: 164

Columns in the dataset:
['task_id', 'prompt', 'entry_point', 'canonical_solution', 'test']

Sample data:
       task_id                                             prompt  \
0  HumanEval/0  from typing import List\n\n\ndef has_close_ele...   
1  HumanEval/1  from typing import List\n\n\ndef separate_pare...   
2  HumanEval/2  \n\ndef truncate_number(number: float) -> floa...   
3  HumanEval/3  from typing import List\n\n\ndef below_zero(op...   
4  HumanEval/4  from typing import List\n\n\ndef mean_absolute...   

               entry_point                                 canonical_solution  \
0       has_close_elements      for idx, elem in enumerate(numbers):\n    ...   
1    separate_paren_groups      result = []\n    current_string = []\n    ...   
2          truncate_number                              return number % 1.0\n   
3               below_zero      balance = 0\n\n    for op in operations:\n...   
4  mean_absolute_deviation      mean = sum(numbers) / l

In [66]:
passed_df

,task_id,completion,result,passed
1,HumanEval/1,groups = []\r\n paren_stack = []\r\n ...,failed: list index out of range,False
5,HumanEval/5,if not numbers:\r\n return []\r\n ...,failed:,False
6,HumanEval/6,"""""""\r\n Algorithm:\r\n Use a sta...",failed: Illegal character,False
10,HumanEval/10,if len(string) == 0:\r\n return str...,failed:,False
12,HumanEval/12,longest_string = ''\r\n longest_string_...,failed:,False
...,...,...,...,...
158,HumanEval/158,max_length = 0\r\n max_str = None\r\n ...,failed: t2,False
159,HumanEval/159,if (number < need):\r\n return [num...,failed: Error,False
160,HumanEval/160,"return sum([int(operand[i])*eval(f""{operat...","failed: invalid syntax (<string>, line 1)",False
161,HumanEval/161,if s.isalpha():\r\n return s.swapca...,failed:,False


In [67]:
df

,task_id,prompt,entry_point,canonical_solution,test
0,HumanEval/0,from typing import List\n\n\ndef has_close_ele...,has_close_elements,"for idx, elem in enumerate(numbers):\n ...","\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
1,HumanEval/1,from typing import List\n\n\ndef separate_pare...,separate_paren_groups,result = []\n current_string = []\n ...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
2,HumanEval/2,\n\ndef truncate_number(number: float) -> floa...,truncate_number,return number % 1.0\n,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
3,HumanEval/3,from typing import List\n\n\ndef below_zero(op...,below_zero,balance = 0\n\n for op in operations:\n...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
4,HumanEval/4,from typing import List\n\n\ndef mean_absolute...,mean_absolute_deviation,mean = sum(numbers) / len(numbers)\n re...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
...,...,...,...,...,...
159,HumanEval/159,"\ndef eat(number, need, remaining):\n """"""\n...",eat,if(need <= remaining):\n return [ n...,def check(candidate):\n\n # Check some simp...
160,HumanEval/160,"\ndef do_algebra(operator, operand):\n """"""\...",do_algebra,expression = str(operand[0])\n for oprt...,def check(candidate):\n\n # Check some simp...
161,HumanEval/161,"\ndef solve(s):\n """"""You are given a string...",solve,flg = 0\n idx = 0\n new_str = list(s...,def check(candidate):\n\n # Check some simp...
162,HumanEval/162,"\ndef string_to_md5(text):\n """"""\n Given...",string_to_md5,import hashlib\n return hashlib.md5(tex...,def check(candidate):\n\n # Check some simp...


In [68]:
failed_ids = passed_df[passed_df['passed'] == False]['task_id'].tolist()
# Or if it's a string: results_df[results_df['passed'] == "FALSE"]['task_id'].tolist()

# 4. Filter the original data to get just the failed questions
failed_questions_df = df[df['task_id'].isin(failed_ids)]
failed_questions_df

,task_id,prompt,entry_point,canonical_solution,test
1,HumanEval/1,from typing import List\n\n\ndef separate_pare...,separate_paren_groups,result = []\n current_string = []\n ...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
5,HumanEval/5,from typing import List\n\n\ndef intersperse(n...,intersperse,if not numbers:\n return []\n\n ...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
6,HumanEval/6,from typing import List\n\n\ndef parse_nested_...,parse_nested_parens,def parse_paren_group(s):\n depth =...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
10,HumanEval/10,\n\ndef is_palindrome(string: str) -> bool:\n ...,make_palindrome,if not string:\n return ''\n\n b...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
12,HumanEval/12,"from typing import List, Optional\n\n\ndef lon...",longest,if not strings:\n return None\n\n ...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
...,...,...,...,...,...
158,HumanEval/158,"\ndef find_max(words):\n """"""Write a functio...",find_max,"return sorted(words, key = lambda x: (-len...",def check(candidate):\n\n # Check some simp...
159,HumanEval/159,"\ndef eat(number, need, remaining):\n """"""\n...",eat,if(need <= remaining):\n return [ n...,def check(candidate):\n\n # Check some simp...
160,HumanEval/160,"\ndef do_algebra(operator, operand):\n """"""\...",do_algebra,expression = str(operand[0])\n for oprt...,def check(candidate):\n\n # Check some simp...
161,HumanEval/161,"\ndef solve(s):\n """"""You are given a string...",solve,flg = 0\n idx = 0\n new_str = list(s...,def check(candidate):\n\n # Check some simp...


In [69]:
!python repress.py --model "meta-llama/CodeLlama-7b-Python-hf" --results_file "failedresults.jsonl"

Loading failed tasks...
Traceback (most recent call last):
  File "/content/human-eval/repress.py", line 183, in <module>
    main()
  File "/content/human-eval/repress.py", line 50, in main
    failed_tasks = load_results(args.results_file)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/human-eval/repress.py", line 26, in load_results
    with open(results_file, 'r') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'failedresults.jsonl'


In [138]:
failed_questions_df

,task_id,prompt,entry_point,canonical_solution,test
26,HumanEval/26,from typing import List\n\n\ndef remove_duplic...,remove_duplicates,import collections\n c = collections.Co...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
46,HumanEval/46,"\n\ndef fib4(n: int):\n """"""The Fib4 number ...",fib4,"results = [0, 0, 2, 0]\n if n < 4:\n ...",\n\nMETADATA = {}\n\n\ndef check(candidate):\n...
68,HumanEval/68,"\ndef pluck(arr):\n """"""\n ""Given an arra...",pluck,if(len(arr) == 0): return []\n evens = ...,def check(candidate):\n\n # Check some simp...
69,HumanEval/69,\ndef search(lst):\n '''\n You are given...,search,frq = [0] * (max(lst) + 1)\n for i in l...,def check(candidate):\n\n # manually genera...
70,HumanEval/70,\ndef strange_sort_list(lst):\n '''\n Gi...,strange_sort_list,"res, switch = [], True\n while lst:\n ...",def check(candidate):\n\n # Check some simp...
82,HumanEval/82,"\ndef prime_length(string):\n """"""Write a fu...",prime_length,l = len(string)\n if l == 0 or l == 1:\...,def check(candidate):\n\n # Check some simp...
88,HumanEval/88,"\ndef sort_array(array):\n """"""\n Given a...",sort_array,return [] if len(array) == 0 else sorted(a...,def check(candidate):\n\n # Check some simp...
95,HumanEval/95,"\ndef check_dict_case(dict):\n """"""\n Giv...",check_dict_case,if len(dict.keys()) == 0:\n return ...,def check(candidate):\n\n # Check some simp...
99,HumanEval/99,\ndef closest_integer(value):\n '''\n Cr...,closest_integer,"from math import floor, ceil\n\n if val...",def check(candidate):\n\n # Check some simp...
127,HumanEval/127,"\ndef intersection(interval1, interval2):\n ...",intersection,def is_prime(num):\n if num == 1 or...,def check(candidate):\n\n # Check some simp...


In [141]:
new_solutions_df

,task_id,prompt,completion
0,HumanEval/26,from typing import List\n\n\ndef remove_duplic...,
1,HumanEval/46,"\n\ndef fib4(n: int):\n """"""The Fib4 number ...","prev_prev = 0\n curr_pprev, curr_prev =..."
2,HumanEval/68,"\ndef pluck(arr):\n """"""\n ""Given an arra...",
3,HumanEval/69,\ndef search(lst):\n '''\n You are given...,
4,HumanEval/70,\ndef strange_sort_list(lst):\n '''\n Gi...,
5,HumanEval/82,"\ndef prime_length(string):\n """"""Write a fu...",
6,HumanEval/88,"\ndef sort_array(array):\n """"""\n Given a...",pass
7,HumanEval/95,"\ndef check_dict_case(dict):\n """"""\n Giv...",
8,HumanEval/99,\ndef closest_integer(value):\n '''\n Cr...,# your code here\n
9,HumanEval/127,"\ndef intersection(interval1, interval2):\n ...",


In [122]:
df

,task_id,prompt,entry_point,canonical_solution,test
0,HumanEval/0,from typing import List\n\n\ndef has_close_ele...,has_close_elements,"for idx, elem in enumerate(numbers):\n ...","\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
1,HumanEval/1,from typing import List\n\n\ndef separate_pare...,separate_paren_groups,result = []\n current_string = []\n ...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
2,HumanEval/2,\n\ndef truncate_number(number: float) -> floa...,truncate_number,return number % 1.0\n,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
3,HumanEval/3,from typing import List\n\n\ndef below_zero(op...,below_zero,balance = 0\n\n for op in operations:\n...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
4,HumanEval/4,from typing import List\n\n\ndef mean_absolute...,mean_absolute_deviation,mean = sum(numbers) / len(numbers)\n re...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
...,...,...,...,...,...
159,HumanEval/159,"\ndef eat(number, need, remaining):\n """"""\n...",eat,if(need <= remaining):\n return [ n...,def check(candidate):\n\n # Check some simp...
160,HumanEval/160,"\ndef do_algebra(operator, operand):\n """"""\...",do_algebra,expression = str(operand[0])\n for oprt...,def check(candidate):\n\n # Check some simp...
161,HumanEval/161,"\ndef solve(s):\n """"""You are given a string...",solve,flg = 0\n idx = 0\n new_str = list(s...,def check(candidate):\n\n # Check some simp...
162,HumanEval/162,"\ndef string_to_md5(text):\n """"""\n Given...",string_to_md5,import hashlib\n return hashlib.md5(tex...,def check(candidate):\n\n # Check some simp...


In [123]:
failed_ids = new_solutions_df[
    (new_solutions_df['completion'] == False) |
    (new_solutions_df['completion'].isna()) |
    (new_solutions_df['completion'] == '') |
    (new_solutions_df['completion'].astype(str).str.isspace()) |
    (new_solutions_df['completion'].astype(str).str.strip() == '')
]['task_id'].tolist()  # Added ['task_id'] here

failed_questions_df = df[df['task_id'].isin(failed_ids)]
failed_questions_df

,task_id,prompt,entry_point,canonical_solution,test
26,HumanEval/26,from typing import List\n\n\ndef remove_duplic...,remove_duplicates,import collections\n c = collections.Co...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
46,HumanEval/46,"\n\ndef fib4(n: int):\n """"""The Fib4 number ...",fib4,"results = [0, 0, 2, 0]\n if n < 4:\n ...",\n\nMETADATA = {}\n\n\ndef check(candidate):\n...
68,HumanEval/68,"\ndef pluck(arr):\n """"""\n ""Given an arra...",pluck,if(len(arr) == 0): return []\n evens = ...,def check(candidate):\n\n # Check some simp...
69,HumanEval/69,\ndef search(lst):\n '''\n You are given...,search,frq = [0] * (max(lst) + 1)\n for i in l...,def check(candidate):\n\n # manually genera...
70,HumanEval/70,\ndef strange_sort_list(lst):\n '''\n Gi...,strange_sort_list,"res, switch = [], True\n while lst:\n ...",def check(candidate):\n\n # Check some simp...
82,HumanEval/82,"\ndef prime_length(string):\n """"""Write a fu...",prime_length,l = len(string)\n if l == 0 or l == 1:\...,def check(candidate):\n\n # Check some simp...
88,HumanEval/88,"\ndef sort_array(array):\n """"""\n Given a...",sort_array,return [] if len(array) == 0 else sorted(a...,def check(candidate):\n\n # Check some simp...
95,HumanEval/95,"\ndef check_dict_case(dict):\n """"""\n Giv...",check_dict_case,if len(dict.keys()) == 0:\n return ...,def check(candidate):\n\n # Check some simp...
99,HumanEval/99,\ndef closest_integer(value):\n '''\n Cr...,closest_integer,"from math import floor, ceil\n\n if val...",def check(candidate):\n\n # Check some simp...
127,HumanEval/127,"\ndef intersection(interval1, interval2):\n ...",intersection,def is_prime(num):\n if num == 1 or...,def check(candidate):\n\n # Check some simp...


In [143]:
import pandas as pd

# Read the CSV file into a pandas DataFrame
df = pd.read_csv('/content/human-eval/humaneval_samples_updated.csv')

# Display the first few rows
df

,passed,completion,result,task_id
0,True,numbers_to_check = sorted(numbers)\n fo...,passed,HumanEval/0
1,False,\n parens = []\n # Stack for keeping tr...,failed:,HumanEval/1
2,True,return number - int(number)\n\n\ndef is_py...,passed,HumanEval/2
3,True,balance = 0\n for operation in operatio...,passed,HumanEval/3
4,True,numbers_mean = sum(numbers) / len(numbers)...,passed,HumanEval/4
...,...,...,...,...
159,False,# TODO - Add Your Code Here - 8 Lines (~3-...,failed: Error,HumanEval/159
160,False,NaN,failed:,HumanEval/160
161,False,\n #for i in range (len(s)): ## Loop ...,failed:,HumanEval/161
162,True,\n import hashlib\n\n if not text:\n ...,passed,HumanEval/162


In [109]:
df

,passed,completion,result,task_id
0,True,numbers_to_check = sorted(numbers)\n fo...,passed,HumanEval/0
1,False,\n parens = []\n # Stack for keeping tr...,failed:,HumanEval/1
2,True,return number - int(number)\n\n\ndef is_py...,passed,HumanEval/2
3,True,balance = 0\n for operation in operatio...,passed,HumanEval/3
4,True,numbers_mean = sum(numbers) / len(numbers)...,passed,HumanEval/4
...,...,...,...,...
159,False,# TODO - Add Your Code Here - 8 Lines (~3-...,failed: Error,HumanEval/159
160,False,NaN,failed:,HumanEval/160
161,False,\n #for i in range (len(s)): ## Loop ...,failed:,HumanEval/161
162,True,\n import hashlib\n\n if not text:\n ...,passed,HumanEval/162


In [145]:
failed_ids = df[df['passed'] == False]['task_id']
len(failed_ids)

103

In [146]:
new_solutions_df

,task_id,prompt,completion
0,HumanEval/26,from typing import List\n\n\ndef remove_duplic...,
1,HumanEval/46,"\n\ndef fib4(n: int):\n """"""The Fib4 number ...","prev_prev = 0\n curr_pprev, curr_prev =..."
2,HumanEval/68,"\ndef pluck(arr):\n """"""\n ""Given an arra...",
3,HumanEval/69,\ndef search(lst):\n '''\n You are given...,
4,HumanEval/70,\ndef strange_sort_list(lst):\n '''\n Gi...,
5,HumanEval/82,"\ndef prime_length(string):\n """"""Write a fu...",
6,HumanEval/88,"\ndef sort_array(array):\n """"""\n Given a...",pass
7,HumanEval/95,"\ndef check_dict_case(dict):\n """"""\n Giv...",
8,HumanEval/99,\ndef closest_integer(value):\n '''\n Cr...,# your code here\n
9,HumanEval/127,"\ndef intersection(interval1, interval2):\n ...",


In [117]:
failed_ids = df_changed[
    (df_changed['completion'] == False) |
    (df_changed['completion'].isna()) |
    (df_changed['completion'] == '') |
    (df_changed['completion'].astype(str).str.isspace()) |
    (df_changed['completion'].astype(str).str.strip() == '')
]['task_id'].tolist()  # Added ['task_id'] here

failed_questions_df = df[df['task_id'].isin(failed_ids)]
failed_questions_df

,task_id,prompt,entry_point,canonical_solution,test
25,HumanEval/25,from typing import List\n\n\ndef factorize(n: ...,factorize,import math\n fact = []\n i = 2\n ...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
26,HumanEval/26,from typing import List\n\n\ndef remove_duplic...,remove_duplicates,import collections\n c = collections.Co...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da..."
33,HumanEval/33,"\n\ndef sort_third(l: list):\n """"""This func...",sort_third,l = list(l)\n l[::3] = sorted(l[::3])\n...,\n\nMETADATA = {}\n\n\ndef check(candidate):\n...
34,HumanEval/34,"\n\ndef unique(l: list):\n """"""Return sorted...",unique,return sorted(list(set(l)))\n,\n\nMETADATA = {}\n\n\ndef check(candidate):\n...
46,HumanEval/46,"\n\ndef fib4(n: int):\n """"""The Fib4 number ...",fib4,"results = [0, 0, 2, 0]\n if n < 4:\n ...",\n\nMETADATA = {}\n\n\ndef check(candidate):\n...
64,HumanEval/64,"\nFIX = """"""\nAdd more test cases.\n""""""\n\ndef ...",vowels_count,"vowels = ""aeiouAEIOU""\n n_vowels = sum(...",def check(candidate):\n\n # Check some simp...
68,HumanEval/68,"\ndef pluck(arr):\n """"""\n ""Given an arra...",pluck,if(len(arr) == 0): return []\n evens = ...,def check(candidate):\n\n # Check some simp...
69,HumanEval/69,\ndef search(lst):\n '''\n You are given...,search,frq = [0] * (max(lst) + 1)\n for i in l...,def check(candidate):\n\n # manually genera...
70,HumanEval/70,\ndef strange_sort_list(lst):\n '''\n Gi...,strange_sort_list,"res, switch = [], True\n while lst:\n ...",def check(candidate):\n\n # Check some simp...
78,HumanEval/78,"\ndef hex_key(num):\n """"""You have been task...",hex_key,"primes = ('2', '3', '5', '7', 'B', 'D')\n ...",def check(candidate):\n\n # Check some simp...




ORIGINAL DF!


In [149]:
import pandas as pd

# Read the CSV file into a pandas DataFrame
df = pd.read_csv('/content/human-eval/humaneval_samples.csv')

# Display the first few rows
df

,completion,task_id
0,numbers_to_check = sorted(numbers)\n fo...,HumanEval/0
1,groups = []\n paren_stack = []\n gro...,HumanEval/1
2,return number - int(number)\n\n\ndef is_py...,HumanEval/2
3,balance = 0\n for operation in operatio...,HumanEval/3
4,numbers_mean = sum(numbers) / len(numbers)...,HumanEval/4
...,...,...
159,if (number < need):\n return [numbe...,HumanEval/159
160,"return sum([int(operand[i])*eval(f""{operat...",HumanEval/160
161,if s.isalpha():\n return s.swapcase...,HumanEval/161
162,\n import hashlib\n\n if not text:\n ...,HumanEval/162


In [115]:
import pandas as pd

# Read the CSV file into a pandas DataFrame
df_changed = pd.read_csv('/content/human-eval/changed.csv')

# Display the first few rows
df_changed

,passed,completion,result,task_id
0,True,numbers_to_check = sorted(numbers)\n fo...,passed,HumanEval/0
1,False,\n parens = []\n # Stack for keeping tr...,failed:,HumanEval/1
2,True,return number - int(number)\n\n\ndef is_py...,passed,HumanEval/2
3,True,balance = 0\n for operation in operatio...,passed,HumanEval/3
4,True,numbers_mean = sum(numbers) / len(numbers)...,passed,HumanEval/4
...,...,...,...,...
159,False,# TODO - Add Your Code Here - 8 Lines (~3-...,failed: Error,HumanEval/159
160,False,NaN,failed:,HumanEval/160
161,False,\n #for i in range (len(s)): ## Loop ...,failed:,HumanEval/161
162,True,\n import hashlib\n\n if not text:\n ...,passed,HumanEval/162


In [135]:
failed_ids = df_changed[df_changed['passed'] == False]['task_id']
len(failed_ids)

103

In [72]:
import pandas as pd

# Read the CSV file into a pandas DataFrame
df = pd.read_csv('/content/human-eval/humaneval_samples.jsonl_results-2.csv')

# Display the first few rows
df

,task_id,completion,result,passed
0,HumanEval/0,numbers_to_check = sorted(numbers)\r\n ...,passed,True
1,HumanEval/1,groups = []\r\n paren_stack = []\r\n ...,failed: list index out of range,False
2,HumanEval/2,return number - int(number)\r\n\r\n\r\ndef...,passed,True
3,HumanEval/3,balance = 0\r\n for operation in operat...,passed,True
4,HumanEval/4,numbers_mean = sum(numbers) / len(numbers)...,passed,True
...,...,...,...,...
159,HumanEval/159,if (number < need):\r\n return [num...,failed: Error,False
160,HumanEval/160,"return sum([int(operand[i])*eval(f""{operat...","failed: invalid syntax (<string>, line 1)",False
161,HumanEval/161,if s.isalpha():\r\n return s.swapca...,failed:,False
162,HumanEval/162,\r\n import hashlib\r\n\r\n if not text:...,passed,True


In [96]:
import json
import csv
import pandas as pd

# Define file paths
jsonl_file_path = '/content/human-eval/humaneval_samples_updated.jsonl_results.jsonl'
csv_file_path = '/content/human-eval/changed.csv'

# Read the JSONL file and convert to a list of dictionaries
data = []
with open(jsonl_file_path, 'r', encoding='utf-8') as jsonl_file:
    for line in jsonl_file:
        if line.strip():  # Skip empty lines
            data.append(json.loads(line))

# If the file is empty, exit early
if not data:
    print("Input file is empty")
else:
    # Get all unique keys to use as CSV headers
    headers = set()
    for item in data:
        headers.update(item.keys())
    headers = list(headers)

    # Write the data to a CSV file
    with open(csv_file_path, 'w', encoding='utf-8', newline='') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=headers)
        writer.writeheader()
        writer.writerows(data)

    print(f"Successfully converted {jsonl_file_path} to {csv_file_path}")

    # Display the first few rows of the CSV file to verify
    df = pd.read_csv(csv_file_path)
    display(df)

Successfully converted /content/human-eval/humaneval_samples_updated.jsonl_results.jsonl to /content/human-eval/changed.csv


,passed,completion,result,task_id
0,True,numbers_to_check = sorted(numbers)\n fo...,passed,HumanEval/0
1,False,\n parens = []\n # Stack for keeping tr...,failed:,HumanEval/1
2,True,return number - int(number)\n\n\ndef is_py...,passed,HumanEval/2
3,True,balance = 0\n for operation in operatio...,passed,HumanEval/3
4,True,numbers_mean = sum(numbers) / len(numbers)...,passed,HumanEval/4
...,...,...,...,...
159,False,# TODO - Add Your Code Here - 8 Lines (~3-...,failed: Error,HumanEval/159
160,False,NaN,failed:,HumanEval/160
161,False,\n #for i in range (len(s)): ## Loop ...,failed:,HumanEval/161
162,True,\n import hashlib\n\n if not text:\n ...,passed,HumanEval/162


In [126]:
import pandas as pd

# Read the CSV file into a pandas DataFrame
df = pd.read_csv('/content/human-eval/changed.csv')

# Display the first few rows
df

,passed,completion,result,task_id
0,True,numbers_to_check = sorted(numbers)\n fo...,passed,HumanEval/0
1,False,\n parens = []\n # Stack for keeping tr...,failed:,HumanEval/1
2,True,return number - int(number)\n\n\ndef is_py...,passed,HumanEval/2
3,True,balance = 0\n for operation in operatio...,passed,HumanEval/3
4,True,numbers_mean = sum(numbers) / len(numbers)...,passed,HumanEval/4
...,...,...,...,...
159,False,# TODO - Add Your Code Here - 8 Lines (~3-...,failed: Error,HumanEval/159
160,False,NaN,failed:,HumanEval/160
161,False,\n #for i in range (len(s)): ## Loop ...,failed:,HumanEval/161
162,True,\n import hashlib\n\n if not text:\n ...,passed,HumanEval/162


In [131]:
new_solutions_df

,task_id,prompt,completion
0,HumanEval/25,from typing import List\n\n\ndef factorize(n: ...,"\n assert n > 0 and isinstance(n, (int)..."
1,HumanEval/26,from typing import List\n\n\ndef remove_duplic...,\n
2,HumanEval/33,"\n\ndef sort_third(l: list):\n """"""This func...",\n result = [] # this will be our final...
3,HumanEval/34,"\n\ndef unique(l: list):\n """"""Return sorted...",return sorted({i for i in l})
4,HumanEval/46,"\n\ndef fib4(n: int):\n """"""The Fib4 number ...",\n
5,HumanEval/64,"\nFIX = """"""\nAdd more test cases.\n""""""\n\ndef ...",pass\n
6,HumanEval/68,"\ndef pluck(arr):\n """"""\n ""Given an arra...",NaN
7,HumanEval/69,\ndef search(lst):\n '''\n You are given...,
8,HumanEval/70,\ndef strange_sort_list(lst):\n '''\n Gi...,
9,HumanEval/78,"\ndef hex_key(num):\n """"""You have been task...",\n # your code goes here\n # HEX_DIG...


In [128]:
new_solutions_df = pd.read_csv('/content/human-eval/regenerated_solutions.csv')
new_solutions_df

,task_id,prompt,completion
0,HumanEval/25,from typing import List\n\n\ndef factorize(n: ...,"\n assert n > 0 and isinstance(n, (int)..."
1,HumanEval/26,from typing import List\n\n\ndef remove_duplic...,\n
2,HumanEval/33,"\n\ndef sort_third(l: list):\n """"""This func...",\n result = [] # this will be our final...
3,HumanEval/34,"\n\ndef unique(l: list):\n """"""Return sorted...",return sorted({i for i in l})
4,HumanEval/46,"\n\ndef fib4(n: int):\n """"""The Fib4 number ...",\n
5,HumanEval/64,"\nFIX = """"""\nAdd more test cases.\n""""""\n\ndef ...",pass\n
6,HumanEval/68,"\ndef pluck(arr):\n """"""\n ""Given an arra...",NaN
7,HumanEval/69,\ndef search(lst):\n '''\n You are given...,
8,HumanEval/70,\ndef strange_sort_list(lst):\n '''\n Gi...,
9,HumanEval/78,"\ndef hex_key(num):\n """"""You have been task...",\n # your code goes here\n # HEX_DIG...


In [150]:
df

,completion,task_id
0,numbers_to_check = sorted(numbers)\n fo...,HumanEval/0
1,groups = []\n paren_stack = []\n gro...,HumanEval/1
2,return number - int(number)\n\n\ndef is_py...,HumanEval/2
3,balance = 0\n for operation in operatio...,HumanEval/3
4,numbers_mean = sum(numbers) / len(numbers)...,HumanEval/4
...,...,...
159,if (number < need):\n return [numbe...,HumanEval/159
160,"return sum([int(operand[i])*eval(f""{operat...",HumanEval/160
161,if s.isalpha():\n return s.swapcase...,HumanEval/161
162,\n import hashlib\n\n if not text:\n ...,HumanEval/162


In [241]:
new_solutions_df

,task_id,prompt,completion
0,HumanEval/25,from typing import List\n\n\ndef factorize(n: ...,\n # Create an empty array for result a...
1,HumanEval/68,"\ndef pluck(arr):\n """"""\n ""Given an arra...",# if not arr:\n # return None\n\n ...
2,HumanEval/70,\ndef strange_sort_list(lst):\n '''\n Gi...,
3,HumanEval/82,"\ndef prime_length(string):\n """"""Write a fu...",
4,HumanEval/95,"\ndef check_dict_case(dict):\n """"""\n Giv...",
5,HumanEval/99,\ndef closest_integer(value):\n '''\n Cr...,# Your code here\n
6,HumanEval/113,"\ndef odd_count(lst):\n """"""Given a list of ...",\n lst2 = [] #...
7,HumanEval/127,"\ndef intersection(interval1, interval2):\n ...",
8,HumanEval/153,"\ndef Strongest_Extension(class_name, extensio...",
9,HumanEval/158,"\ndef find_max(words):\n """"""Write a functio...",max = -1; curr= ''\n \n# for i in ran...


In [256]:
import pandas as pd

# Load both CSV files
# Assuming the files are saved as humaneval_samples.csv (164 rows) and new_solutions_df.csv (109 rows)
original_df = new_df.copy()
print(f"Original dataframe: {len(original_df)} rows")
print(f"New solutions dataframe: {len(new_solutions_df)} rows")

# Create a mapping from task_id to completion in the new solutions dataframe
completion_map = dict(zip(new_solutions_df['task_id'], new_solutions_df['completion']))

# Function to replace completion if task_id exists in the new solutions
def replace_completion(row):
    task_id = row['task_id']
    if task_id in completion_map:
        return completion_map[task_id]
    return row['completion']

# Create a copy of the original dataframe
updated_df = original_df.copy()

# Apply the replacement function
updated_df['completion'] = updated_df.apply(replace_completion, axis=1)

# Count how many completions were replaced
replacements_count = sum(1 for task_id in original_df['task_id'] if task_id in completion_map)
print(f"Replaced {replacements_count} completions in the original dataframe")

# Save the updated dataframe
updated_df.to_csv('/content/human-eval/update5.csv', index=False)
print(f"Updated dataframe saved to '/content/human-eval/update5.csv'")



Original dataframe: 164 rows
New solutions dataframe: 5 rows
Replaced 5 completions in the original dataframe
Updated dataframe saved to '/content/human-eval/update5.csv'


In [243]:
updated_df

,task_id,completion,prompt
0,HumanEval/0,numbers_to_check = sorted(numbers)\n fo...,from typing import List\n\n\ndef has_close_ele...
1,HumanEval/1,\n parens = []\n # Stack for keeping tr...,from typing import List\n\n\ndef separate_pare...
2,HumanEval/2,return number - int(number)\n\n\ndef is_py...,\n\ndef truncate_number(number: float) -> floa...
3,HumanEval/3,balance = 0\n for operation in operatio...,from typing import List\n\n\ndef below_zero(op...
4,HumanEval/4,numbers_mean = sum(numbers) / len(numbers)...,from typing import List\n\n\ndef mean_absolute...
...,...,...,...
159,HumanEval/159,# TODO - Add Your Code Here - 8 Lines (~3-...,"\ndef eat(number, need, remaining):\n """"""\n..."
160,HumanEval/160,op=operator[0]; operand1=operand[0]; opera...,"\ndef do_algebra(operator, operand):\n """"""\..."
161,HumanEval/161,\n #for i in range (len(s)): ## Loop ...,"\ndef solve(s):\n """"""You are given a string..."
162,HumanEval/162,\n import hashlib\n\n if not text:\n ...,"\ndef string_to_md5(text):\n """"""\n Given..."


In [257]:
# updated_df = pd.read_csv('/content/human-eval/humaneval_samples_updated.csv')

updated_df = updated_df[['task_id', 'completion']]
updated_df

,task_id,completion
0,HumanEval/0,numbers_to_check = sorted(numbers)\n fo...
1,HumanEval/1,\n parens = []\n # Stack for keeping tr...
2,HumanEval/2,return number - int(number)\n\n\ndef is_py...
3,HumanEval/3,balance = 0\n for operation in operatio...
4,HumanEval/4,numbers_mean = sum(numbers) / len(numbers)...
...,...,...
159,HumanEval/159,# TODO - Add Your Code Here - 8 Lines (~3-...
160,HumanEval/160,op=operator[0]; operand1=operand[0]; opera...
161,HumanEval/161,\n #for i in range (len(s)): ## Loop ...
162,HumanEval/162,\n import hashlib\n\n if not text:\n ...


In [258]:
import csv
import json

# Define file paths
csv_file_path = '/content/human-eval/update5.csv'
jsonl_file_path = '/content/human-eval/update5.jsonl'

# Read the CSV file and convert to a list of dictionaries
data = []
with open(csv_file_path, 'r', encoding='utf-8') as csv_file:
    csv_reader = csv.DictReader(csv_file)
    for row in csv_reader:
        data.append(row)

# Write the data to a JSONL file
with open(jsonl_file_path, 'w', encoding='utf-8') as jsonl_file:
    for item in data:
        # Convert each row to a JSON string and write to file
        jsonl_file.write(json.dumps(item) + '\n')

print(f"Successfully converted {csv_file_path} to {jsonl_file_path}")

# Display some statistics
print(f"Converted {len(data)} records from CSV to JSONL")

Successfully converted /content/human-eval/update5.csv to /content/human-eval/update5.jsonl
Converted 164 records from CSV to JSONL


In [261]:
!python3 results.py /content/human-eval/update5.jsonl_results.jsonl

Reading results from: /content/human-eval/update5.jsonl_results.jsonl

===== RESULTS SUMMARY =====
Total problems: 164
Passed: 62
Failed: 102
Pass rate: 37.80%

===== DETAILED RESULTS =====
HumanEval/0: ✓ PASSED
HumanEval/1: ✗ FAILED
HumanEval/2: ✓ PASSED
HumanEval/3: ✓ PASSED
HumanEval/4: ✓ PASSED
HumanEval/5: ✗ FAILED
   Error: assignment expression cannot rebind comprehension iteration variable 'i' (<string>, line 14)
HumanEval/6: ✗ FAILED
HumanEval/7: ✓ PASSED
HumanEval/8: ✓ PASSED
HumanEval/9: ✓ PASSED
HumanEval/10: ✗ FAILED
HumanEval/11: ✓ PASSED
HumanEval/12: ✗ FAILED
   Error: maximum recursion depth exceeded while calling a Python object
HumanEval/13: ✓ PASSED
HumanEval/14: ✓ PASSED
HumanEval/15: ✓ PASSED
HumanEval/16: ✓ PASSED
HumanEval/17: ✓ PASSED
HumanEval/18: ✓ PASSED
HumanEval/19: ✗ FAILED
   Error: closing parenthesis ')' does not match opening parenthesis '[' (<string>, line 22)
HumanEval/20: ✓ PASSED
HumanEval/21: ✓ PASSED
HumanEval/22: ✓ PASSED
HumanEval/23: ✓ PASSED

In [259]:
!evaluate_functional_correctness update5.jsonl

Reading samples...
164it [00:00, 5303.47it/s]
Running test suites...
100% 164/164 [00:04<00:00, 38.99it/s]
Writing results to update5.jsonl_results.jsonl...
100% 164/164 [00:00<00:00, 37631.48it/s]
{'pass@1': 0.3780487804878049}


In [262]:
!pwd

/content/human-eval


In [265]:
%cd /content/human-eval
!pwd

/content/human-eval
/content/human-eval


In [29]:
!zip -r directory_contents.zip .

  adding: model_completions.jsonl_results.jsonl (deflated 85%)
  adding: data/ (stored 0%)
  adding: data/example_samples.jsonl (deflated 66%)
  adding: data/example_problem.jsonl (deflated 29%)
  adding: data/HumanEval.jsonl.gz (deflated 0%)
  adding: README.md (deflated 51%)
  adding: update5.csv (deflated 70%)
  adding: humaneval_samples2.jsonl_results.jsonl (deflated 91%)
  adding: human_eval.egg-info/ (stored 0%)
  adding: human_eval.egg-info/PKG-INFO (deflated 24%)
  adding: human_eval.egg-info/SOURCES.txt (deflated 57%)
  adding: human_eval.egg-info/dependency_links.txt (stored 0%)
  adding: human_eval.egg-info/top_level.txt (deflated 18%)
  adding: human_eval.egg-info/requires.txt (stored 0%)
  adding: human_eval.egg-info/entry_points.txt (deflated 31%)
  adding: model_completions.jsonl (deflated 85%)
  adding: setup.py (deflated 49%)
  adding: results.py (deflated 57%)
  adding: update5_with_combined_prompts.csv (deflated 74%)
  adding: humaneval_samples1.jsonl_results.jsonl (

GENERATED CODE, OG PROMPT, CT PROMPT

In [1]:
import pandas as pd

df = pd.read_csv('/content/update5.csv')
df

,task_id,completion,prompt
0,HumanEval/0,numbers_to_check = sorted(numbers)\n fo...,from typing import List\n\n\ndef has_close_ele...
1,HumanEval/1,\n parens = []\n # Stack for keeping tr...,from typing import List\n\n\ndef separate_pare...
2,HumanEval/2,return number - int(number)\n\n\ndef is_py...,\n\ndef truncate_number(number: float) -> floa...
3,HumanEval/3,balance = 0\n for operation in operatio...,from typing import List\n\n\ndef below_zero(op...
4,HumanEval/4,numbers_mean = sum(numbers) / len(numbers)...,from typing import List\n\n\ndef mean_absolute...
...,...,...,...
159,HumanEval/159,# TODO - Add Your Code Here - 8 Lines (~3-...,"\ndef eat(number, need, remaining):\n """"""\n..."
160,HumanEval/160,op=operator[0]; operand1=operand[0]; opera...,"\ndef do_algebra(operator, operand):\n """"""\..."
161,HumanEval/161,\n #for i in range (len(s)): ## Loop ...,"\ndef solve(s):\n """"""You are given a string..."
162,HumanEval/162,\n import hashlib\n\n if not text:\n ...,"\ndef string_to_md5(text):\n """"""\n Given..."


In [2]:
import pandas as pd

# Read the existing CSV file
df = pd.read_csv('/content/update5.csv')

# Get the CTPROMPT value from your variable
# From your screenshot, it appears to be defined as: CTPROMPT = ""
# You should replace this with the actual value from your environment
CTPROMPT = "There might be an error in the code below because of lack of understanding of the question. Please correct the error, if any, and rewrite the solution. Only output the final correct Python program! \n"  # Replace with your actual CTPROMPT value

# Print information about the DataFrame
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Using CTPROMPT value: '{CTPROMPT}'")

# Create a new combined prompt column
df['combined_prompt'] = df.apply(
    lambda row: CTPROMPT + str(row['prompt']) + str(row['completion']),
    axis=1
)

# # Show the first few rows to verify
# print("\nFirst few rows with combined prompt:")
# sample_columns = ['task_id', 'combined_prompt']
# pd.set_option('display.max_colwidth', 80)  # Limit display width for readability
# print(df[sample_columns].head())

# # Save the updated DataFrame
# output_file = '/content/human-eval/update5_with_combined_prompts.csv'
# df.to_csv(output_file, index=False)
# print(f"\nSaved updated DataFrame to: {output_file}")

# # If you want to create a new DataFrame with just specific columns
# # For example, if you only need task_id and the combined prompt
# selected_columns = ['task_id', 'combined_prompt']
# new_df = df[selected_columns]

# # Save the new DataFrame
# new_output_file = '/content/human-eval/task_id_with_combined_prompts.csv'
# new_df.to_csv(new_output_file, index=False)
# print(f"Also saved simplified version with just {', '.join(selected_columns)} to: {new_output_file}")

# # Print a more detailed example of one combined prompt
# print("\nDetailed example for first row:")
# first_row = df.iloc[0]
# print(f"Task ID: {first_row['task_id']}")
# print(f"CTPROMPT component: '{CTPROMPT}'")
# print(f"Completion component: '{first_row['completion'][:50]}...'")
# print(f"Prompt component: '{first_row['prompt'][:50]}...'")
# print(f"Combined result: '{first_row['combined_prompt'][:100]}...'")

DataFrame shape: (164, 3)
Columns: ['task_id', 'completion', 'prompt']
Using CTPROMPT value: 'There might be an error in the code below because of lack of understanding of the question. Please correct the error, if any, and rewrite the solution. Only output the final correct Python program! 
'


In [5]:
df = df[['task_id', 'combined_prompt']]
output_file = '/content/update5_with_combined_prompts.csv'
df.to_csv(output_file, index=False)

In [5]:
import pandas as pd

# Read the existing CSV file
df = pd.read_csv('/content/human-eval/update5_with_combined_prompts.csv')
df

,task_id,combined_prompt
0,HumanEval/0,There might be an error in the code below beca...
1,HumanEval/1,There might be an error in the code below beca...
2,HumanEval/2,There might be an error in the code below beca...
3,HumanEval/3,There might be an error in the code below beca...
4,HumanEval/4,There might be an error in the code below beca...
...,...,...
159,HumanEval/159,There might be an error in the code below beca...
160,HumanEval/160,There might be an error in the code below beca...
161,HumanEval/161,There might be an error in the code below beca...
162,HumanEval/162,There might be an error in the code below beca...


In [ ]:
inputs = tokenizer(prompt2, return_tensors="pt",
      padding=True,
      return_attention_mask=True).to(model.device)
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=1024,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

In [ ]:
model_name = "meta-llama/CodeLlama-7b-Python-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16
)

In [8]:
from human_eval.data import read_problems, write_jsonl
import itertools
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# Step 1: Load the HumanEval problems
problems = read_problems()

# Step 2: Get the first 10 problem IDs
first_10_keys = list(itertools.islice(problems.keys(), 5))

# Step 3: Load your model and tokenizer
model_name = "meta-llama/CodeLlama-7b-Python-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16
)
model.to("cuda" if torch.cuda.is_available() else "cpu")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096

In [ ]:
from human_eval.data import read_problems, write_jsonl
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
import pandas as pd
import gc

model_name = "meta-llama/CodeLlama-7b-Python-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
        # Set pad token to a different token than eos_token to avoid warning
        tokenizer.pad_token = "[PAD]"
        # Add this token to the vocabulary
        tokenizer.add_special_tokens({"pad_token": "[PAD]"})

# Update token embeddings to account for any added tokens
model.resize_token_embeddings(len(tokenizer))

In [14]:
df_subset = df.head(5)
df_subset

,task_id,combined_prompt
0,HumanEval/0,There might be an error in the code below beca...
1,HumanEval/1,There might be an error in the code below beca...
2,HumanEval/2,There might be an error in the code below beca...
3,HumanEval/3,There might be an error in the code below beca...
4,HumanEval/4,There might be an error in the code below beca...


In [17]:
prompt_column = 'combined_prompt'

samples = []

# Set batch size (adjust based on your GPU memory)
batch_size = 1  # Start with 1 and increase if memory allows

# Process dataframe in batches to improve efficiency
for i in tqdm(range(0, len(df), batch_size), desc="Processing batches"):
    batch_df = df.iloc[i:i+batch_size]

    for _, row in batch_df.iterrows():
        problem_id = row['task_id']
        prompt = row[prompt_column]

        # Skip if prompt is missing
        if pd.isna(prompt):
            print(f"Warning: Prompt is NaN for task {problem_id}, skipping")
            continue

        # Convert prompt to string if it's not already
        prompt = str(prompt)

        # Print the prompt for the first few samples (debugging)
        if i < 2:
            print(f"\nTask ID: {problem_id}")
            print(f"Prompt (first 100 chars): {prompt[:100]}...")

        # Tokenize the prompt
        inputs = tokenizer(prompt, return_tensors="pt", padding=True, return_attention_mask=True).to(model.device)

        # Generate text
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=1024,
                temperature=0.7,
                top_p=0.95,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        # Decode the generated text
        generated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract just the completion part (everything after the prompt)
        completion = generated_code[len(prompt):]

        # Add the result to our samples
        samples.append({
            "task_id": problem_id,
            "completion": completion
        })

    # Clear CUDA cache periodically to avoid OOM errors
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

# Write results to a JSONL file
output_file = '/content/human-eval/model_completions.jsonl'
write_jsonl(output_file, samples)
print(f"\nWrote {len(samples)} samples to {output_file}")

# Also save as CSV for easier inspection
output_csv = '/content/human-eval/model_completions.csv'
pd.DataFrame(samples).to_csv(output_csv, index=False)
print(f"Also saved results as CSV to {output_csv}")

# Display a few examples of the results
print("\nSample completions:")
for i, sample in enumerate(samples[:2]):  # Show first 2 samples
    print(f"\nTask ID: {sample['task_id']}")
    print(f"Completion (first 200 chars): {sample['completion'][:200]}...")

Processing batches:   0%|          | 0/164 [00:00<?, ?it/s]


Task ID: HumanEval/0
Prompt (first 100 chars): There might be an error in the code below because of lack of understanding of the question. Please c...


Processing batches:   1%|          | 1/164 [00:00<01:16,  2.14it/s]


Task ID: HumanEval/1
Prompt (first 100 chars): There might be an error in the code below because of lack of understanding of the question. Please c...


Processing batches: 100%|██████████| 164/164 [50:14<00:00, 18.38s/it]


Wrote 164 samples to /content/human-eval/model_completions.jsonl
Also saved results as CSV to /content/human-eval/model_completions.csv

Sample completions:

Task ID: HumanEval/0
Completion (first 200 chars): ...

Task ID: HumanEval/1
Completion (first 200 chars): 

if __name__ == "__main__":
    import doctest
    doctest.testmod()

    print(separate_paren_groups("( ) (( )) (( )( ))"))
...


In [18]:
!evaluate_functional_correctness model_completions.jsonl

Reading samples...
164it [00:00, 4277.08it/s]
Running test suites...
100% 164/164 [00:01<00:00, 122.32it/s]
Writing results to model_completions.jsonl_results.jsonl...
100% 164/164 [00:00<00:00, 43134.50it/s]
{'pass@1': 0.012195121951219513}


In [22]:
!python3 results.py /content/human-eval/model_completions.jsonl_results.jsonl

Reading results from: /content/human-eval/model_completions.jsonl_results.jsonl

===== RESULTS SUMMARY =====
Total problems: 164
Passed: 2
Failed: 162
Pass rate: 1.22%

===== DETAILED RESULTS =====
HumanEval/0: ✗ FAILED
HumanEval/1: ✗ FAILED
HumanEval/2: ✗ FAILED
HumanEval/3: ✗ FAILED
HumanEval/4: ✗ FAILED
   Error: unsupported operand type(s) for -: 'NoneType' and 'float'
HumanEval/5: ✗ FAILED
HumanEval/6: ✗ FAILED
   Error: unexpected indent (<string>, line 13)
HumanEval/7: ✗ FAILED
HumanEval/8: ✗ FAILED
HumanEval/9: ✗ FAILED
HumanEval/10: ✗ FAILED
   Error: '[' was never closed (<string>, line 128)
HumanEval/11: ✗ FAILED
   Error: unterminated triple-quoted string literal (detected at line 83) (<string>, line 67)
HumanEval/12: ✗ FAILED
   Error: '[' was never closed (<string>, line 42)
HumanEval/13: ✗ FAILED
HumanEval/14: ✗ FAILED
HumanEval/15: ✗ FAILED
   Error: expected ':' (<string>, line 124)
HumanEval/16: ✗ FAILED
HumanEval/17: ✗ FAILED
HumanEval/18: ✗ FAILED
HumanEval/19: ✗ FA

In [ ]:
completions = pd.read_csv('/content/human-eval/model_completions.csv')
completions

In [23]:
model_name

'meta-llama/CodeLlama-7b-Python-hf'

In [24]:
from human_eval.data import read_problems, write_jsonl
import itertools
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

problems = read_problems()

# model_name = "meta-llama/Llama-3.2-3B"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# tokenizer.pad_token = tokenizer.eos_token


# # Apply 4-bit quantization
# # bnb_config = BitsAndBytesConfig(
# #     load_in_4bit=True,
# #     bnb_4bit_compute_dtype=torch.float16,
# #     bnb_4bit_use_double_quant=True,
# #     bnb_4bit_quant_type="nf4"
# # )

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     #quantization_config=bnb_config,
#     device_map="auto"
# )

samples_first = []
samples_second = []
for problem_id in tqdm(problems.keys()):
    #  Prompt 1
    coding_prompt = problems[problem_id]["prompt"]
    prompt1_header = "You are an expert Python programmer, and here is your task: Complete the following python function: \n"
    prompt1 = prompt1_header + coding_prompt
    input1 = tokenizer(prompt1, return_tensors="pt").to(model.device)

    output1 = model.generate(
        input1.input_ids,
        max_length=500,
        temperature=0.2,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_code1 = tokenizer.decode(output1[0], skip_special_tokens=True)

    completion1 = generated_code1[len(prompt1):] # remove header and coding prompt

    # Prompt 2
    prompt2_header = "There might be an error in the code below because of lack of understanding of the question. Please correct the error, if any, and rewrite the solution. Only output the final correct Python program! \n"
    prompt2 = prompt2_header + generated_code1[len(prompt1_header):] # adds just the coding portion of prompt 1's response

    input2 = tokenizer(prompt2, return_tensors="pt").to(model.device)

    output2 = model.generate(
        input2.input_ids,
        max_length=1000,
        temperature=0.2,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_code2 = tokenizer.decode(output2[0], skip_special_tokens=True)

    completion2 = generated_code2[len(prompt2_header) + len(coding_prompt):] # remove header and coding prompt

    samples_first.append({
        "task_id": problem_id,
        "completion": completion1
    })

    samples_second.append({
        "task_id": problem_id,
        "completion": completion2
    })

write_jsonl("humaneval_samples1.jsonl", samples_first)
write_jsonl("humaneval_samples2.jsonl", samples_second)

100%|██████████| 164/164 [1:06:22<00:00, 24.28s/it]


In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

In [25]:
!evaluate_functional_correctness humaneval_samples1.jsonl

Reading samples...
164it [00:00, 7456.54it/s]
Running test suites...
100% 164/164 [00:03<00:00, 50.34it/s] 
Writing results to humaneval_samples1.jsonl_results.jsonl...
100% 164/164 [00:00<00:00, 54359.56it/s]
{'pass@1': 0.27439024390243905}


In [26]:
!evaluate_functional_correctness humaneval_samples2.jsonl

Reading samples...
164it [00:00, 7162.14it/s]
Running test suites...
100% 164/164 [00:03<00:00, 50.25it/s]
Writing results to humaneval_samples2.jsonl_results.jsonl...
100% 164/164 [00:00<00:00, 37965.88it/s]
{'pass@1': 0.2865853658536585}


In [27]:
import json

file_path = "humaneval_samples1.jsonl_results.jsonl"
data = []

with open(file_path, "r") as f:
    for line in f:
        data.append(json.loads(line.strip()))

data_dict1 = {i: item for i, item in enumerate(data)}

file_path = "humaneval_samples2.jsonl_results.jsonl"
data = []

with open(file_path, "r") as f:
    for line in f:
        data.append(json.loads(line.strip()))

data_dict2 = {i: item for i, item in enumerate(data)}

In [28]:
# Get Results
count_i_c = 0
count_c_i = 0
correct_1 = 0
correct_2 = 0
num_problems = len(problems)
for i in range(len(data_dict1)):
    if data_dict1[i]['passed']:
        correct_1 += 1
    if data_dict2[i]['passed']:
        correct_2 += 1
    if data_dict1[i]['passed'] and not data_dict2[i]['passed']:
        count_c_i += 1
    elif not data_dict1[i]['passed'] and data_dict2[i]['passed']:
        count_i_c += 1

print("Accuracy@t1: " + str(correct_1 / num_problems))
print("Accuracy@t2: " + str(correct_2 / num_problems))
print("delta(t1,t2): " + str((correct_2 - correct_1) / num_problems))
print("delta(t1,t2) i to c: " + str(count_i_c / num_problems))
print("delta(t2,t1) c to i: " + str(count_c_i / num_problems))

Accuracy@t1: 0.27439024390243905
Accuracy@t2: 0.2865853658536585
delta(t1,t2): 0.012195121951219513
delta(t1,t2) i to c: 0.018292682926829267
delta(t2,t1) c to i: 0.006097560975609756
